# 🧠 Sistema de Inteligência Pessoal
### Análise de Dados Comportamentais

**Como usar:**
1. Exporte sua planilha do Google Sheets como CSV (`Arquivo → Fazer download → CSV`)
2. Execute a célula de instalação abaixo
3. Faça upload do CSV quando solicitado
4. Execute as demais células em ordem

---

In [ ]:
# ⚙️ CÉLULA 1 — Instalação de dependências
!pip install pandas numpy matplotlib seaborn -q

In [ ]:
# 📦 CÉLULA 2 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from google.colab import files
import io
import warnings
warnings.filterwarnings('ignore')

# Estilo visual
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')
print('✅ Imports prontos')

In [ ]:
# 📂 CÉLULA 3 — Upload do CSV
print('📂 Selecione o arquivo CSV exportado do Google Sheets...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f'✅ Arquivo carregado: {filename}')
print(f'📊 {len(df_raw)} dias de dados, {len(df_raw.columns)} colunas')
df_raw.head(3)

In [ ]:
# 🧹 CÉLULA 4 — Limpeza e preparação dos dados

df = df_raw.copy()

# Remover colunas vazias e irrelevantes
colunas_remover = ['Carimbo de data/hora', 'Column 23', 'Column 24']
df = df.drop(columns=[c for c in colunas_remover if c in df.columns], errors='ignore')

# Coluna de data
data_col = [c for c in df.columns if 'Data' in c or 'data' in c][0]
df = df.rename(columns={data_col: 'Data'})
df['Data'] = pd.to_datetime(df['Data'], dayfirst=True)
df = df.sort_values('Data').reset_index(drop=True)

# Remover colunas de texto livre (Sintoma) da análise numérica
colunas_texto = ['Sintoma']
df_texto = df[['Data'] + [c for c in colunas_texto if c in df.columns]].copy()
df = df.drop(columns=[c for c in colunas_texto if c in df.columns], errors='ignore')

# Garantir que todas as métricas são numéricas
metricas = [c for c in df.columns if c != 'Data']
for col in metricas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'✅ Dados limpos: {len(df)} dias | {len(metricas)} variáveis analíticas')
print(f'📅 Período: {df["Data"].min().strftime("%d/%m/%Y")} → {df["Data"].max().strftime("%d/%m/%Y")}')
print(f'\n📋 Variáveis: {metricas}')

In [ ]:
# 📊 CÉLULA 5 — Visão geral dos dados

print('═' * 50)
print('📊 RESUMO ESTATÍSTICO')
print('═' * 50)

resumo = df[metricas].describe().round(2).T[['mean', 'std', 'min', 'max']]
resumo.columns = ['Média', 'Desvio', 'Mínimo', 'Máximo']
resumo = resumo.sort_values('Média', ascending=False)

display(resumo)

# Dados faltantes
faltantes = df[metricas].isnull().sum()
if faltantes.sum() > 0:
    print(f'\n⚠️ Dados faltantes:')
    print(faltantes[faltantes > 0])
else:
    print('\n✅ Nenhum dado faltante')

In [ ]:
# 🔥 CÉLULA 6 — Heatmap de Correlações

corr = df[metricas].corr()

fig, ax = plt.subplots(figsize=(14, 10))

mascara = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr,
    mask=mascara,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 8}
)

ax.set_title('🔥 Matriz de Correlação — Todas as Variáveis', 
             fontsize=14, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('correlacoes.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Heatmap salvo como correlacoes.png')

In [ ]:
# 🎯 CÉLULA 7 — Top fatores por variável-alvo

alvos = ['Energia Geral', 'Humor', 'Produtividade']
alvos = [a for a in alvos if a in metricas]  # só os que existem nos dados

fig, axes = plt.subplots(1, len(alvos), figsize=(6 * len(alvos), 6))
if len(alvos) == 1:
    axes = [axes]

for ax, alvo in zip(axes, alvos):
    correlacoes = corr[alvo].drop(alvo).sort_values()
    cores = ['#e74c3c' if v < 0 else '#2ecc71' for v in correlacoes]
    
    correlacoes.plot(kind='barh', ax=ax, color=cores)
    ax.set_title(f'Fatores que impactam\n{alvo}', fontweight='bold', fontsize=11)
    ax.set_xlabel('Correlação')
    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.axvline(x=0.3, color='green', linewidth=0.5, linestyle='--', alpha=0.5)
    ax.axvline(x=-0.3, color='red', linewidth=0.5, linestyle='--', alpha=0.5)
    
    for i, (val, name) in enumerate(zip(correlacoes, correlacoes.index)):
        ax.text(val + (0.02 if val >= 0 else -0.02), i, 
                f'{val:.2f}', va='center', 
                ha='left' if val >= 0 else 'right', fontsize=8)

plt.suptitle('🎯 O que mais impacta cada variável-alvo', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fatores_impacto.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo como fatores_impacto.png')

In [ ]:
# 📈 CÉLULA 8 — Séries temporais das métricas principais

principais = ['Energia Geral', 'Humor', 'Produtividade', 'Clareza Mental']
principais = [p for p in principais if p in metricas]

fig, ax = plt.subplots(figsize=(14, 5))

cores_linhas = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']

for i, col in enumerate(principais):
    ax.plot(df['Data'], df[col], 
            marker='o', linewidth=2, markersize=5,
            label=col, color=cores_linhas[i % len(cores_linhas)])

ax.set_ylim(-0.2, 5.2)
ax.set_yticks([0, 1, 2, 3, 4, 5])
ax.axhline(y=2.5, color='gray', linestyle='--', alpha=0.4, linewidth=0.8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
plt.xticks(rotation=45)
ax.set_title('📈 Evolução das Métricas Principais ao Longo do Tempo', 
             fontweight='bold', fontsize=12)
ax.set_xlabel('Data')
ax.set_ylabel('Score (0–5)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('series_temporais.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo como series_temporais.png')

In [ ]:
# 💡 CÉLULA 9 — Insights automáticos em texto

print('═' * 55)
print('💡 INSIGHTS AUTOMÁTICOS')
print('═' * 55)

for alvo in alvos:
    print(f'\n🎯 {alvo.upper()}')
    corrs_alvo = corr[alvo].drop(alvo).sort_values(ascending=False)
    
    positivos = corrs_alvo[corrs_alvo >= 0.3]
    negativos = corrs_alvo[corrs_alvo <= -0.3]
    
    if len(positivos) > 0:
        print(f'  ✅ Aumenta quando:')
        for var, val in positivos.items():
            forca = 'forte' if abs(val) >= 0.6 else 'moderada'
            print(f'     → {var} está alto (correlação {forca}: {val:.2f})')
    else:
        print('  ✅ Nenhuma correlação positiva forte ainda (mais dados necessários)')
    
    if len(negativos) > 0:
        print(f'  ⚠️  Reduz quando:')
        for var, val in negativos.items():
            forca = 'forte' if abs(val) >= 0.6 else 'moderada'
            print(f'     → {var} está alto (correlação {forca}: {val:.2f})')
    else:
        print('  ⚠️  Nenhuma correlação negativa forte ainda (mais dados necessários)')

print('\n' + '═' * 55)
print(f'📊 Base atual: {len(df)} dias')
print(f'📌 Correlações confiáveis a partir de ~30 dias de dados')
print('═' * 55)

In [ ]:
# 💾 CÉLULA 10 — Exportar dados limpos e correlações

# CSV limpo
df.to_csv('dados_limpos.csv', index=False)

# Correlações em formato flat (para BI futuro)
corr_flat = corr.stack().reset_index()
corr_flat.columns = ['Variavel_1', 'Variavel_2', 'Correlacao']
corr_flat = corr_flat[
    (corr_flat['Variavel_1'] != corr_flat['Variavel_2']) &
    (corr_flat['Correlacao'].abs() >= 0.1)
].sort_values('Correlacao', ascending=False)
corr_flat.to_csv('correlacoes_flat.csv', index=False)

# Download automático
files.download('dados_limpos.csv')
files.download('correlacoes_flat.csv')
files.download('correlacoes.png')
files.download('fatores_impacto.png')
files.download('series_temporais.png')

print('✅ Arquivos exportados:')
print('   📄 dados_limpos.csv — dados normalizados')
print('   📄 correlacoes_flat.csv — para BI futuro')
print('   🖼️  correlacoes.png — heatmap')
print('   🖼️  fatores_impacto.png — fatores por variável')
print('   🖼️  series_temporais.png — evolução no tempo')